<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: Preprocesamiento y Análisis de Datos (Landslide4Sense)
**Estudiante:** Lisa | Maestría en Ingeniería - EAFIT

Este notebook se conecta a Google Drive y realiza el análisis exploratorio de las 14 bandas multiespectrales.

## 1. Conexión y Configuración de Rutas
Montamos Drive y localizamos los archivos .h5 de forma recursiva.

In [ ]:
from google.colab import drive
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Montar Drive
drive.mount('/content/drive')

# 2. Búsqueda profunda de datos (Soluciona el error de 0 muestras)
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    # Definimos las rutas finales basadas en el hallazgo real
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    
    print(f"✅ ¡Éxito! Datos encontrados en: {train_img_dir}")
    print(f"Número de imágenes (H5): {len(img_list)}")
    print(f"Número de máscaras (H5): {len(mask_list)}")
else:
    print("❌ ERROR: No se encontró la carpeta 'TrainData/img' dentro de Landslide4Sense.")
    print("Estructura actual en Drive:")
!ls -R /content/drive/MyDrive/Landslide4Sense | head -n 15

## 2. Análisis Visual Multiespectral
Visualizamos Sentinel-2 (RGB), Pendiente (Slope) y Elevación (DEM).

In [ ]:
def load_h5(file_path):
    with h5py.File(file_path, 'r') as f:
        return np.array(f[list(f.keys())[0]])

if len(img_list) > 0:
    idx = np.random.randint(0, len(img_list))
    img = load_h5(img_list[idx])
    mask = load_h5(mask_list[idx])
    
    # RGB: Bandas 4, 3, 2 (índices 3, 2, 1)
    rgb = img[:, :, [3, 2, 1]]
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    
    # Capas Geotécnicas: Banda 13 (Slope), Banda 14 (DEM)
    slope = img[:, :, 12]
    dem = img[:, :, 13]
    
    fig, ax = plt.subplots(1, 4, figsize=(22, 5))
    ax[0].imshow(rgb); ax[0].set_title("Sentinel-2 (RGB)"); ax[0].axis('off')
    ax[1].imshow(slope, cmap='terrain'); ax[1].set_title("Mapa de Pendientes"); ax[1].axis('off')
    ax[2].imshow(dem, cmap='gist_earth'); ax[2].set_title("Elevación (DEM)"); ax[2].axis('off')
    ax[3].imshow(mask, cmap='magma'); ax[3].set_title("Zonas de Deslizamiento"); ax[3].axis('off')
    plt.show()
else:
    print("No hay datos cargados para visualizar.")

## 3. Estadísticas de Clases
Cálculo del desbalance de píxeles para el reporte de tesis.

In [ ]:
def check_balance(limit=50):
    clase_0, clase_1 = 0, 0
    for i in range(min(limit, len(mask_list))):
        m = load_h5(mask_list[i])
        clase_1 += np.sum(m == 1)
        clase_0 += np.sum(m == 0)
    
    total = clase_0 + clase_1
    print(f"Análisis de balance (píxeles):")
    print(f"- Sin deslizamiento: {clase_0/total:.2%}")
    print(f"- Con deslizamiento: {clase_1/total:.2%}")

check_balance()